# 50 Train DPO Qwen3.5 4B

DPO stage configuration using the current generic TRL `DPOTrainer` path.

Required preference mix:
- identity preferences (25k input contract)
- ultrafeedback cleaned (primary)
- optional additional preferences disabled by default


In [ ]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import yaml

from lumis1.hashing import sha256_file
from lumis1.run_evidence import create_run_evidence_tree, write_run_status, write_run_summary

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load((REPO_ROOT / "configs" / "train_dpo.yaml").read_text(encoding="utf-8"))
profile_cfg = yaml.safe_load((REPO_ROOT / "configs" / "run_profiles.yaml").read_text(encoding="utf-8"))

PROFILE = "default_96gb"
RUN_ID = "manual-dpo"
RUN_TRAINING = False

if cfg["model"]["dtype"].lower() != "bf16":
    raise RuntimeError("DPO should run in BF16 mode")
if cfg["model"].get("load_in_4bit") is not False:
    raise RuntimeError("4-bit training is disabled for Qwen3.5")
if cfg["preferences"]["optional_additional_preferences"]["enabled"]:
    raise RuntimeError("optional additional preferences must be disabled by default")

profile = profile_cfg["profiles"][PROFILE]["dpo"]
DPO_OUTPUT_DIR = REPO_ROOT / cfg["outputs"]["run_dir"]
resolved = {
    "profile": PROFILE,
    "run_id": RUN_ID,
    "model": cfg["model"],
    "lora": cfg["lora"],
    "training": dict(cfg["training"], **profile),
    "dpo": cfg["dpo"],
    "preferences": cfg["preferences"],
    "run_training": RUN_TRAINING,
    "output_dir": str(DPO_OUTPUT_DIR),
}

out = REPO_ROOT / cfg["outputs"]["resolved_config_report"]
out.write_text(json.dumps(resolved, indent=2), encoding="utf-8")

run_paths = None
if RUN_TRAINING:
    run_paths = create_run_evidence_tree(REPO_ROOT, RUN_ID)
    (run_paths["config_snapshot"] / "train_dpo_resolved.json").write_text(
        json.dumps(resolved, indent=2),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_50_invocation.txt").write_text(
        "Notebook 50 manual execution for DPO training.\n"
        f"profile={PROFILE}\n"
        f"output_dir={DPO_OUTPUT_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(
        REPO_ROOT,
        RUN_ID,
        stage="dpo",
        status="running",
        details={"profile": PROFILE, "output_dir": str(DPO_OUTPUT_DIR)},
    )

print(json.dumps(resolved, indent=2))
print(f"saved: {out}")


In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING is False. No DPO execution.")
else:
    from datasets import load_dataset
    from trl import DPOConfig, DPOTrainer
    from transformers import AutoModelForCausalLM, AutoTokenizer

    pref_path = REPO_ROOT / "workspace" / "final" / "full_preferences.jsonl"
    ds = load_dataset("json", data_files=str(pref_path), split="train")

    try:
        model = AutoModelForCausalLM.from_pretrained(cfg["model"]["sft_checkpoint_or_adapter"])
        tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["sft_checkpoint_or_adapter"])
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        def format_prompt(example: dict[str, object]) -> dict[str, object]:
            prompt = example.get("prompt")
            if not isinstance(prompt, str) or not prompt.strip():
                prompt_messages = example.get("prompt_messages")
                if isinstance(prompt_messages, list):
                    prompt_parts = [
                        str(item.get("content", "")).strip()
                        for item in prompt_messages
                        if isinstance(item, dict)
                        and item.get("role") == "user"
                        and isinstance(item.get("content"), str)
                        and str(item.get("content", "")).strip()
                    ]
                    prompt = " ".join(prompt_parts).strip()
            if not isinstance(prompt, str) or not prompt.strip():
                raise RuntimeError("Preference row missing prompt or prompt_messages")
            rendered_prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            if "<think>" in rendered_prompt or "</think>" in rendered_prompt:
                raise RuntimeError("Rendered DPO prompt still contains thinking markers")
            return {"prompt": rendered_prompt}

        ds = ds.map(format_prompt)
        trainer = DPOTrainer(
            model=model,
            ref_model=None,
            processing_class=tokenizer,
            train_dataset=ds,
            args=DPOConfig(**resolved["training"], output_dir=str(DPO_OUTPUT_DIR)),
            beta=float(cfg["dpo"]["beta"]),
        )
        train_result = trainer.train()
        trainer.save_model(str(DPO_OUTPUT_DIR))
        tokenizer.save_pretrained(str(DPO_OUTPUT_DIR))

        train_metrics = getattr(train_result, "metrics", {})
        checksums = {
            str(path.relative_to(DPO_OUTPUT_DIR)): sha256_file(path)
            for path in sorted(DPO_OUTPUT_DIR.rglob("*"))
            if path.is_file()
        }
        report = {
            "run_id": RUN_ID,
            "output_dir": str(DPO_OUTPUT_DIR),
            "metrics": train_metrics,
            "artifact_files": sorted(checksums),
        }
        if run_paths is not None:
            (run_paths["reports"] / "dpo_training.json").write_text(
                json.dumps(report, indent=2),
                encoding="utf-8",
            )
            (run_paths["checksums"] / "artifacts.json").write_text(
                json.dumps(checksums, indent=2),
                encoding="utf-8",
            )
            write_run_status(REPO_ROOT, RUN_ID, stage="dpo", status="completed", details=report)
            write_run_summary(
                REPO_ROOT,
                RUN_ID,
                "# DPO Summary\n\nThis summary reflects notebook 50 execution.\n\n```json\n"
                + json.dumps(report, indent=2)
                + "\n```",
            )
        print(json.dumps(report, indent=2))
    except Exception as exc:
        if run_paths is not None:
            write_run_status(
                REPO_ROOT,
                RUN_ID,
                stage="dpo",
                status="failed",
                details={"error": str(exc), "output_dir": str(DPO_OUTPUT_DIR)},
            )
        raise
